# Lab 05. 선형회귀와 예측 오차

선형회귀는 다음 형태의 예측식을 학습한다.

$$\hat y=\beta_0+\beta_1x_1+\cdots+\beta_px_p$$

최소제곱법은 잔차제곱합을 최소화한다.

$$SSE=\sum_i(y_i-\hat y_i)^2$$

이 Notebook에서는 단순회귀 기울기를 직접 계산한 뒤 다중회귀와 테스트 데이터 평가로 확장한다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 루트에서 Notebook을 실행하세요.")

sys.path.insert(0, str(ROOT))
STUDENT_ID = "20260001"  # 반드시 본인 학번으로 변경
print("저장소:", ROOT)
print("실습 학번:", STUDENT_ID)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.education.personalized_data import make_student_dataset, student_seed

df = make_student_dataset(STUDENT_ID).drop_duplicates()
for column in ["유동인구", "월임대료"]:
    df[column] = df[column].fillna(df[column].median())

## 1. 완성 예제: 단순회귀 계수 직접 계산

단순회귀 기울기와 절편은 다음과 같다.

$$\beta_1=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{\sum(x_i-\bar x)^2},
\qquad \beta_0=\bar y-\beta_1\bar x$$

In [ ]:
x = df["유동인구"].to_numpy()
y = df["월매출"].to_numpy()
beta1 = ((x-x.mean()) * (y-y.mean())).sum() / ((x-x.mean())**2).sum()
beta0 = y.mean() - beta1 * x.mean()
manual_pred = beta0 + beta1 * x

simple = LinearRegression().fit(x.reshape(-1, 1), y)
print("직접 계산:", beta0, beta1)
print("sklearn:", simple.intercept_, simple.coef_[0])
assert np.isclose(beta1, simple.coef_[0])

기울기는 유동인구가 1단위 증가할 때 예측 월매출이 평균적으로 얼마나 달라지는지를 뜻한다.
관찰 연구이므로 유동인구 증가가 매출 증가의 원인이라고 단정할 수 없다.

## 2. 학습·테스트 분할

테스트 데이터는 모델 선택과 학습에 사용하지 않는다. random_state를 학번 시드로 고정하면 재현할 수 있다.

In [ ]:
features = ["유동인구", "경쟁점포수", "주차장수", "월임대료"]
X = df[features]
y = df["월매출"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=student_seed(STUDENT_ID)
)
print("학습:", X_train.shape, "테스트:", X_test.shape)

## 3. 다중회귀와 평가

MAE는 평균 절대오차, RMSE는 큰 오차에 더 민감한 제곱 기반 오차다. R²는 평균 예측과 비교한 설명 비율이다.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** .5
r2 = r2_score(y_test, pred)
coefficients = pd.Series(model.coef_, index=features, name="회귀계수")
display(coefficients.to_frame())
print(f"MAE={mae:.1f}, RMSE={rmse:.1f}, R2={r2:.3f}")

## 4. 기준모델과 비교

복잡한 모델이 의미 있으려면 최소한 학습 데이터 평균만 예측하는 기준모델보다 좋아야 한다.

In [ ]:
baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = mean_squared_error(y_test, baseline_pred) ** .5
print("기준 MAE/RMSE:", baseline_mae, baseline_rmse)
print("회귀 MAE/RMSE:", mae, rmse)

## 5. 잔차 분석

잔차는 실제값-예측값이다. 예측값에 따른 곡선 패턴이나 깔때기 모양은 선형성·등분산 가정을 의심하게 한다.

In [ ]:
residual = y_test.to_numpy() - pred
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, pred, alpha=.7)
limits = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(limits, limits, "--", color="red")
axes[0].set(xlabel="실제 월매출", ylabel="예측 월매출", title="실제값과 예측값")

axes[1].scatter(pred, residual, alpha=.7)
axes[1].axhline(0, linestyle="--", color="red")
axes[1].set(xlabel="예측 월매출", ylabel="잔차", title="잔차 진단")
plt.tight_layout()
plt.show()

## 6. 독립 연습

1. 업종을 one-hot encoding하여 모델에 추가하고 테스트 성능을 비교한다.
2. 이상 후보를 포함한 모델과 제외한 모델의 MAE·RMSE를 비교한다.
3. 학습·테스트 분할 시드를 세 개 바꾸어 성능 변동을 확인한다.
4. 계수의 단위와 부호를 설명하고 상식과 다른 계수의 가능한 원인을 제안한다.

In [ ]:
# TODO: get_dummies와 ColumnTransformer 중 하나를 이용해 업종 포함 모델 작성
extended_result = None
display(extended_result)

# TODO: 세 random_state의 평가표
stability_table = None
display(stability_table)

## 7. 자가점검

- [ ] 단순회귀 기울기를 공식으로 계산했다.
- [ ] 테스트 데이터는 학습에 사용하지 않았다.
- [ ] 기준모델과 MAE·RMSE를 비교했다.
- [ ] 잔차 그래프에서 패턴과 이상값을 확인했다.
- [ ] 예측 관계를 인과관계로 표현하지 않았다.